In [30]:
import pandas as pd
import librosa 
from pathlib import Path
import numpy as np 
import soundfile as sf
from IPython.display import Audio
import scipy.signal as sig
from tqdm import tqdm 


## Create windowing function

Need to fade onsets and offsets of cue and mixture to avoid popping

In [31]:
%matplotlib inline
import matplotlib.pyplot as plt

In [32]:
## Define windowing function - will apply a cosine ramp to the start and end of a signal

def ramp_hann(x, ramp_dur_ms, samplerate=20_000):
    stim_dur_smp = x.shape[0] # N taps of x
    ramp_dur_smp =  np.floor(ramp_dur_ms * samplerate / 1000).astype('int')
    assert stim_dur_smp > (2*ramp_dur_smp), 'Ramps cannot be longer than the stimulus duration'
    
    # calc window
    # https://stackoverflow.com/questions/56485663/hanning-window-values-doesnt-match-in-python-and-matlab
    win = sig.hann((2 * ramp_dur_smp) + 2)[1:-1]
    
    # Beginning of windowed stimulus
    start_win = win[ : ramp_dur_smp ] * x[ : ramp_dur_smp]
    
    # Middle part (steady state)
    steady_win = x[ramp_dur_smp : stim_dur_smp-ramp_dur_smp]
    
    # Final part of windowed stimulus
    end_win = win[ramp_dur_smp : ramp_dur_smp*2] * x[stim_dur_smp-ramp_dur_smp:stim_dur_smp]

    return np.hstack([start_win, steady_win, end_win])

def rms_normalize(wav, new_rms=0.07, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    wav = wav * new_rms / rms_wav
    return wav


# Generate stim for each dataset 

To make stimuli, combine cue, isi, and mixture for each trial into a wav form, and save it to our server.  

Naming convention:
`/full/path/<set_ix>_<ix_in_pandas>_<noise_cond>_<snr>_<word_int>_<speaker_sex>.wav`

This will let us map the audio back to the rows in its parent pandas dataframe

#### Generate trial audio data for each attentive listening task dataset 

Get path to pandas dataframes 

In [33]:
df_paths = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes_mturk/').glob('*.pdpkl'))

In [34]:
len(df_paths) # should be 25 

25

In [35]:
## Define default params

sampling_rate = 20_000 # 20kHz
sig_ramp_dur = 10 # in ms 
isi_in_frames = int(0.25 * sampling_rate) # 250ms at 20kHz

isi_array = np.zeros(isi_in_frames)

out_dir = Path('/mindhive/mcdermott/www/mturk_stimuli/imgriff/timit_attentive_task_mturk/') 

# Save files to condition directories 

one directory for each distractor x snr condition
e.g.: `/timit_attentive_listening_task/one_distractor_-5dB_SNR`  or `/timit_attentive_listening_task/ssn_distractor_10dB_SNR`

file names inside directory have mapping back to pandas df and relevant labeling for analysis:
`<set_ix>_<ix_in_pandas>_<noise_cond>_<snr>_<word_int>_<speaker_sex>.wav`

In [36]:
distractor_map = {1:'one',
                 2:'two',
                 4:'four'}

In [37]:
for df_ix, df_path in tqdm(enumerate(df_paths)):
    if 'training' in df_path.as_posix():
        continue 
    dataset = pd.read_pickle(df_path)
    # loop over trials
    for trial in range(len(dataset)):
        # get cue and mixture signal
        cue, mixture = dataset.loc[trial, ['cue_signal', 'mixture_signal']]
        noise_cond = dataset['distractor_conditions'][trial]
        snr = dataset['snrs'][trial]
        word_int = dataset['word_int'][trial]
        speaker_sex = dataset['speaker_sex'][trial]
        # fade cue and mixture
        cue = ramp_hann(cue, sig_ramp_dur)
        mixture = ramp_hann(mixture, sig_ramp_dur)
        # combine as trial audio
        trial_wav = np.hstack([cue, isi_array, mixture])
        # get dir name for stimulus
        if noise_cond == 'catch trial':
            dir_name = 'catch_trials'
            stim_name = f"set_{df_ix:02d}_stim_{trial:03d}_word_{word_int}_speaker_{speaker_sex}.wav"
        else:
            noise_name = distractor_map[int(noise_cond)] if noise_cond != 'ssn' else 'ssn'
            dir_name = f"{noise_name}_distractor_{snr}dB_snr"
            stim_name = f"set_{df_ix:02d}_stim_{trial:03d}_cond_{noise_cond}_snr_{snr}_word_{word_int}_speaker_{speaker_sex}.wav"
        
        dataset_out_path = out_dir / dir_name
        if not dataset_out_path.exists():
            dataset_out_path.mkdir(parents=True, exist_ok=True)
        out_name = dataset_out_path / stim_name
        # write file 
        sf.write(out_name.as_posix(), trial_wav, sampling_rate, subtype='PCM_16') 

    

25it [00:19,  1.31it/s]


In [38]:
noise_cond

'catch trial'

### Save training stimulti 

In [39]:
training_stim_file = '/om2/user/imgriff/datasets/timit/attn_task_dataframes/attn_task_training_excerpts_0_1rms_mturk.pdpkl'

train_set = pd.read_pickle(training_stim_file)


In [40]:
train_set.head()

,_original_timit_index,word,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,signal,word_int,cue_signal,cue_word,_original_cue_timit_index,cue_speaker
0,23,novel,train-dr1-fdaw0-sx326,fdaw0,20000,40000,f,sx,sx326,dr1,"[0.011782246730868976, 0.08233421110820222, 0....",461,"[-0.08059113233432856, -0.1574949309033704, -0...",an,14,fdaw0
1,7554,become,test-dr1-felc0-sx306,felc0,20000,40000,f,sx,sx306,dr1,"[-0.149552760494615, -0.18440199070296268, -0....",76,"[-0.07331340547644653, -0.07308784018363555, -...",greasy,7543,felc0
2,4376,difficult,train-dr5-fgdp0-sx448,fgdp0,20000,40000,f,sx,sx448,dr5,"[0.00139015216211521, 0.0007251573614166049, 0...",203,"[-0.007427113597063375, -0.014920978401971072,...",heroic,4367,fgdp0
3,9180,house,test-dr5-fhes0-sx389,fhes0,20000,40000,f,sx,sx389,dr5,"[0.0002804323043918055, 0.0006113662873129717,...",321,"[0.16474580322706292, 0.21244699506143175, 0.2...",a,9167,fhes0
4,7566,quite,test-dr1-fjem0-si634,fjem0,20000,40000,f,si,si634,dr1,"[0.03028287467191142, 0.036875799966606865, 0....",566,"[0.01904497844170738, 0.027698903427361518, 0....",more,7573,fjem0


In [41]:
train_set.columns

Index(['_original_timit_index', 'word', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'signal', 'word_int', 'cue_signal', 'cue_word',
       '_original_cue_timit_index', 'cue_speaker'],
      dtype='object')

In [42]:
dir_name = "feedback_trials_train_data"

for trial in range(len(train_set)):
    # get cue and mixture signal
    cue, signal = train_set.loc[trial, ['cue_signal', 'signal']]
    word_int = train_set['word_int'][trial]
    speaker_sex = train_set['speaker_sex'][trial]
    # fade cue and mixture
    cue = ramp_hann(cue, sig_ramp_dur)
    signal = ramp_hann(signal, sig_ramp_dur)
    # combine as trial audio
    trial_wav = np.hstack([cue, isi_array, signal])

    dataset_out_path = out_dir / dir_name
    if not dataset_out_path.exists():
        dataset_out_path.mkdir(parents=True, exist_ok=True)
    # <set_ix>_<ix_in_pandas>_<noise_cond>_<snr>_<word_int>_<speaker_sex>.wav
    stim_name = f"train_set_stim_{trial:02d}_word_{word_int}_speaker_{speaker_sex}.wav"
    out_name = dataset_out_path / stim_name
    # write file 
    sf.write(out_name.as_posix(), trial_wav, sampling_rate, subtype='PCM_16') 



In [43]:
wav, fs = sf.read(out_name)

In [ ]:
Audio(out_name)